In [ ]:
!git clone https://github.com/sakjin2/Final-Project-Intro-to-AI-ML
import os
os.chdir('Final-Project-Intro-to-AI-ML')

In [ ]:
!pip install "transformers<5.0.0" torch faiss-cpu sentence-transformers gradio pdfplumber google-genai -q


In [ ]:
import torch
import numpy as np
import faiss
import textwrap
from transformers import pipeline

In [ ]:
print(f'Device: {"GPU" if torch.cuda.is_available() else "CPU"}')
''' change device to 0 in the next cell if you have GPU '''

In [ ]:
from transformers import pipeline
embedder = pipeline(
    "feature-extraction",
    model="sentence-transformers/all-MiniLM-L6-v2",
    tokenizer="sentence-transformers/all-MiniLM-L6-v2",
    device=-1
)
EMBED_DIM = 384


In [ ]:
def embed_text(text: str) -> np.ndarray:
    raw=embedder(text)
    new_Raw = np.array(raw[0])
    vec =new_Raw.mean(axis=0)
    return vec.astype(np.float32)

In [ ]:
import pdfplumber
import os
document_path = [path for path in os.listdir("pdf") if path.endswith('.pdf')]
document_full_path = [os.path.join("pdf",path) for path in document_path]
def extract_text(path: str) ->str:
  text=""
  with pdfplumber.open(path) as pdf:
    for page in pdf.pages:
      pagetext=page.extract_text()
      if pagetext:
        text+=pagetext+"\n"
  return text
documents=[extract_text(path) for path in document_full_path]


In [ ]:
def chunk_documents(documents: list[str],document_path:list[str] ,chunk_size: int = 200,overlap: int = 40,) -> list[tuple[str,str]]:
    chunks = []
    for doc,path in zip(documents,document_path):
      for i in range(0, len(doc), chunk_size-overlap):
        chunk = doc[i : i + chunk_size]
        if chunk:
          chunks.append((chunk, path))
    return chunks
chunks = chunk_documents(documents,document_path)


In [ ]:
def build_index(chunks: list[tuple[str,str]]) -> tuple[faiss.Index, np.ndarray]:
    chunk_embeddings=np.array([embed_text(c[0]) for c in chunks])
    faiss.normalize_L2(chunk_embeddings)
    index = faiss.IndexFlatIP(EMBED_DIM)
    index.add(chunk_embeddings)
    return index,chunk_embeddings
index, chunk_embeddings = build_index(chunks)

In [ ]:
def retrieve(question: str,index: faiss.Index,chunks: list[tuple[str,str]],k: int = 3,threshhold: float = 0.25) -> list[tuple[tuple[str,str], float]]:
    question_embedding = embed_text(question)
    question_embedding = question_embedding.reshape(1,EMBED_DIM).astype('float32')
    faiss.normalize_L2(question_embedding)
    scores, indices = index.search(question_embedding,k)
    return [(chunks[idx], float(score)) for idx, score in zip(indices[0], scores[0]) if score>=threshhold]

In [ ]:
def build_prompt(question: str, context_chunks: list[tuple[tuple[str,str]],float]) -> str:
  context="\n\n".join(
        f"[Source {i+1}]: {chunk[0]}" for i, (chunk,score) in enumerate(context_chunks)
        )
  prompt = f'''
You are an IITB Academic Assistant,an assistant that answers questions about IIT Bombay using Institute documents.

Answer the question directly and naturally,as if you simply know the answer-do not say phrases like "according to the provided text","based on the context",or "the document says".If you do need to reference where information comes from,say "according to institute documents" instead.

If the answer is not present in the information below, respond only with: "I don't know."

Institute documents:
{context}
Question: {question}
Answer:'''
  return prompt


In [ ]:
import os
from google import genai
from google.genai import types
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')


In [ ]:
client = genai.Client()
def generator(prompt, max_new_tokens=150):
    config = types.GenerateContentConfig(max_output_tokens=max_new_tokens)
    response = client.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents=prompt,
        config=config
    )
    return response.text






In [ ]:
def generate_answer(prompt: str, max_new_tokens: int = 4000) -> str:
  result = generator(prompt, max_new_tokens=max_new_tokens)
  return result

In [ ]:
def rag_answer(question: str, index: faiss.Index, chunks: list[tuple[str,str]], k: int = 3) -> dict:
  context_chunks = retrieve(question , index ,chunks,k)
  if not context_chunks:
    return {
        "answer": "I don't know.",
        "sources": "Sources for this result are:\nNone-no relevant documents found."
      }
  prompt = build_prompt(question , context_chunks)
  result = generate_answer(prompt)
  sources = set([str(c[0][1]) for c in context_chunks])
  source_str = "Sources for this result are:\n"
  source_str += ",".join(sources)
  return {
        "answer": result.strip(),
        "sources": source_str
    }

In [ ]:
import gradio as gr
def interface(question,history):
  result = rag_answer(question,index,chunks,5)
  return f"{result['answer']}\n\n{result['sources']}"
demo = gr.ChatInterface(fn = interface, title = "Academic Assistant" , description = "Ask doubts about academics at IITB")
demo.launch(share=True)
